In [ ]:
!pip uninstall -qqy jupyterlab

In [ ]:
!pip install -qU "google-genai==1.7.0" "chromadb==0.6.3"

In [ ]:
from google import genai
from google.genai import types

from IPython.display import Markdown

genai.__version__

'1.7.0'

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

In [ ]:
documents = []

import os

base_dir = '/content'

for filename in os.listdir(base_dir):
    if filename.endswith('.txt'):
        file_path = os.path.join(base_dir, filename)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            documents.append(content)

len(documents)

8

In [ ]:
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.api_core import retry

from google.genai import types

is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

In [ ]:
class GeminiEmbeddingFunction(EmbeddingFunction):
    # Specify whether to generate embeddings for documents, or queries
    document_mode = True

    @retry.Retry(predicate=is_retriable)
    def __call__(self, input: Documents) -> Embeddings:
        if self.document_mode:
            embedding_task = "retrieval_document"
        else:
            embedding_task = "retrieval_query"

        response = client.models.embed_content(
            model="models/text-embedding-004",
            contents=input,
            config=types.EmbedContentConfig(
                task_type=embedding_task,
            ),
        )
        return [e.values for e in response.embeddings]

In [ ]:
client = genai.Client(api_key=GOOGLE_API_KEY)

In [ ]:
import chromadb

DB_NAME = "googlecardb"

embed_fn = GeminiEmbeddingFunction()
embed_fn.document_mode = True

chroma_client = chromadb.Client()
db = chroma_client.get_or_create_collection(name=DB_NAME, embedding_function=embed_fn)

db.add(documents=documents, ids=[str(i) for i in range(len(documents))])

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
db.count()
# db.peek(1)

8

In [ ]:
def GuardianAI(query: str) -> str:
    embed_fn.document_mode = False

    # Search the Chroma DB using the specified query.

    result = db.query(query_texts=[query], n_results=1)
    [all_passages] = result["documents"]
    query_oneline = query.replace("\n", " ")

    prompt = f"""You are a helpful and informative bot that answers query of new students related to IIT BHU and Varanasi City related
    questions using text from the reference passage included below.
    Be sure to respond in a complete sentence, being comprehensive, including all relevant background information.
    However, you may or may not be talking to a non-technical audience, so be sure to break down complicated concepts and
    strike a friendly and converstional tone. If the passage is irrelevant to the answer, you may tell the user that "the database is not yet updated to answer your query".

    QUESTION: {query_oneline}
    """

    for passage in all_passages:
        passage_oneline = passage.replace("\n", " ")
        prompt += f"PASSAGE: {passage_oneline}\n"


    response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt)

    return response.text

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create a text input widget for the user query
query_box = widgets.Text(
    placeholder='Enter your query...',
    description='Query:',
    disabled=False
)

# Create a button widget for submission
submit_button = widgets.Button(
    description='Submit',
    button_style='primary'
)

# Create an output widget to display the processed output
output = widgets.Output()

# Define the event handler for the button click
def on_submit_clicked(button):
    # Clear any previous output
    with output:
        clear_output()
        # Get the query from the text input widget
        query = query_box.value
        # Process the query using the custom function
        result = GuardianAI(query)
        # Display the result
        print(result)

# Bind the click event of the button to the handler
submit_button.on_click(on_submit_clicked)

# Display the widgets in the notebook
display(query_box, submit_button, output)


Text(value='', description='Query:', placeholder='Enter your query...')

Button(button_style='primary', description='Submit', style=ButtonStyle())

Output()

In [ ]:
!pip install pyngrok --quiet
from pyngrok import ngrok

# Terminate open tunnels if they exist
ngrok.kill()

# Set the authtoken (replace '<YOUR_NGROK_AUTH_TOKEN>' with your actual token)
ngrok.set_auth_token("2vS4Sz8jHXPbMfTXlftuPZo7Sw2_ZAj6popH2qmncfkC3pP8")

# Open an HTTPS tunnel on port 5000
public_url = ngrok.connect(5000, bind_tls=True)
print("Public URL:", public_url)


In [ ]:
!pip install flask-cors

In [ ]:
from flask_cors import CORS


In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

CORS(app, origins=["http://localhost:5173"], supports_credentials=True)

@app.route('/guadian-ai', methods=['POST','OPTIONS'])
def handle_query():
    if request.method == 'OPTIONS':
        response = app.make_response('')
        response.headers.add("Access-Control-Allow-Origin", "http://localhost:5173")
        response.headers.add("Access-Control-Allow-Headers", "Content-Type")
        response.headers.add("Access-Control-Allow-Methods", "POST, OPTIONS")
        response.headers.add("Access-Control-Allow-Credentials", "true")
        return response, 200
    data = request.get_json()
    query = data.get('query', '')
    response_text = GuardianAI(query)
    return jsonify({'response': response_text})

if __name__ == '__main__':
    app.run(port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
